# SFINCS — NJ Sandy: results & methodology viewer

A **visualization-only** companion to the build/run pipeline. The model is built
and run by the experiment harness (`run_experiments.py` + the `nj_sfincs`
package); this notebook just **opens a finished run read-only and plots it**, so
you can show your advisor any methodological choice on demand — the elevation
model, the grid, the mask, each forcing, the flood map, and the validation
against the Sandy Hook gauge / USGS High Water Marks / the FEMA MOTF extent.

The last section compares every wave experiment side by side.

> Pick which run to view by setting **`EXP`** in the setup cell. It falls back to
> the reference `model/` build if that experiment hasn't been run yet.

## Setup

In [ ]:
# Import the viz stack up top (this also primes PROJ before hydromt loads).
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# Make the nj_sfincs package importable from notebooks/.
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from hydromt_sfincs import SfincsModel
from nj_sfincs import plots, validate
from nj_sfincs.config import EXPERIMENTS

# ── Choose the run to view ───────────────────────────────────────────────────
EXP = "snapwave_tuned"  # any of: baseline_no_waves, wind_waves, snapwave_tuned, igwaves, igwaves_wind, wavemaker
exp_dir = ROOT / "experiments" / EXP
if not exp_dir.exists():
    exp_dir = ROOT / "model"  # fall back to the reference build
    print(f"experiments/{EXP} not found — showing the reference model/ build")
print("viewing:", exp_dir)


In [ ]:
# Open the run twice: `sf` (read mode, for the build + forcing inputs) and
# `mod` (for the solver output), and downscale the flood map once.
DATA_LIBS = [str(ROOT / "data" / "data_catalog.yml")]
sf = SfincsModel(str(exp_dir), data_libs=DATA_LIBS, mode="r")
sf.read()   # repopulates grid + every forcing component from disk

mod, da_hmax, da_dep = validate.load_floodmap(exp_dir)
print("output vars:", list(mod.output.data.keys()))

---
## Methodology — the static build

These document the modeling choices: the quadtree grid, the merged topobathy,
and the active/boundary mask.

### The quadtree grid

In [ ]:
plots.plot_grid(sf);

### Topobathy (interactive) — pan/zoom the dunes, inlets, dredged channels

In [ ]:
plots.plot_topobathy(sf)

### The mask — active interior, water-level boundary, outflow

In [ ]:
plots.plot_mask(sf);

---
## Methodology — the forcing

The compound drivers, read back from the written model: the observed surge
boundary, ERA5 wind/pressure, AORC rainfall, and USGS river discharge.

### Surge boundary (NOAA CO-OPS)

In [ ]:
plots.plot_surge(sf);

### Wind + pressure (ERA5)

In [ ]:
plots.plot_wind_pressure(sf);

### Rainfall (NOAA AORC)

In [ ]:
plots.plot_rain(sf);

### River discharge (USGS)

In [ ]:
plots.plot_discharge(sf);

---
## Results

The SnapWave field (if this run has waves), the downscaled flood map, and the
three validations.

### SnapWave Hm0 at peak — look for a lee behind Sandy Hook

In [ ]:
res = plots.plot_wave_field(mod)
if res is None:
    print("no wave output for this run (waves off, or no hm0)")

### Maximum flood depth

In [ ]:
plots.plot_floodmap(mod, da_hmax);

### Validation 1 — Sandy Hook gauge (temporal)

In [ ]:
m = validate.gauge_peak_error(mod)
print(f"observed peak: {m['gauge_obs_peak_m']:.2f} m | modeled peak: "
      f"{m['gauge_mod_peak_m']:.2f} m | error: {m['gauge_peak_err_m']:+.2f} m")

### Validation 2 — USGS High Water Marks (spatial)

In [ ]:
plots.plot_hwm_scatter(da_hmax, da_dep);

In [ ]:
plots.plot_hwm_residual_map(mod, da_hmax, da_dep);

### Validation 3 — FEMA MOTF extent (CSI / POD / FAR)

In [ ]:
plots.plot_motf(da_hmax, da_dep);

---
## Compare the wave experiments

Reads `experiments/metrics.csv` (written by `run_experiments.py`). Higher CSI /
POD and lower FAR / HWM-RMSE are better; `shb_hm0` is the SnapWave Hm0 in the
Sandy Hook Bay lee — the "did waves reach the bay?" number.

A self-contained **`experiments/report.html`** with the same table + flood-map
thumbnails is written by the runner — that's the file to email your advisor.

In [ ]:
metrics_csv = ROOT / "experiments" / "metrics.csv"
if metrics_csv.exists():
    metrics = pd.read_csv(metrics_csv, index_col=0)
    display(metrics.round(3))
else:
    metrics = None
    print("No experiments/metrics.csv yet — run:  python run_experiments.py")

In [ ]:
if metrics is not None:
    plots.plot_experiment_comparison(metrics, ROOT / "experiments" / "floodmaps");

---

## Engine comparison — Faber vs Galibier (Workstream I)

Two SFINCS builds disagree about how much water reaches the Shrewsbury/Navesink,
and the disagreement is **entirely wave setup**: with `snapwave=0` both engines give
an identical Shrewsbury peak of **1.663 m**, so the barotropic core is not in play.

| run | engine | `gammax` | `bexp` | Shrewsbury peak |
|---|---|---|---|---|
| `galibier_nowaves_25m` | v2.4.0 | – | – | 1.663 m |
| `snapwave_tuned_25m` | v2.3.3 Faber | 2 (clamped) | 0 | 2.223 m |
| `galibier_gammax2_bexp0_25m` | v2.4.0 Galibier | 2 (clamped) | 0 | **3.096 m** |
| `galibier_gammax2_25m` | v2.4.0 Galibier | 2 (clamped) | 2 | 3.380 m |

Observed crest at USGS 01407600 is **2.935 m**. `snapwave_gammax` is the per-sweep
wave-height clamp — Faber defaults it to 2, Galibier ships it as 999 (disabled),
which is what made the stock Galibier blow up. With the clamp restored, both engines
are stable *and* converged (`niter` becomes inert), so the ~1.2 m spread below is
physics, not numerics.

**Do not pick a winner from proximity to 2.935 m.** The 2-anchor surge boundary
(Atlantic City + The Battery) carries a bias of order ±0.3 m at this latitude, which
is the same size as the gap being judged.

In [ ]:
# Cached-tif reader, so this is seconds rather than a re-downscale per run.
# Any run missing its floodmap_hmax_lev3.tif renders as a "not downscaled yet" panel;
# call validate.load_floodmap(ROOT / "experiments" / run) once to build it.
ENGINE_RUNS = {
    "galibier_nowaves_25m":       "waves OFF (barotropic)\nShrewsbury 1.663 m",
    "snapwave_tuned_25m":         "FABER  clamp2 bexp0\nShrewsbury 2.223 m",
    "galibier_gammax2_bexp0_25m": "GALIBIER clamp2 bexp0\nShrewsbury 3.096 m",
    "galibier_gammax2_25m":       "GALIBIER clamp2 bexp2\nShrewsbury 3.380 m",
}
plots.plot_engine_panels(ENGINE_RUNS);

### Where the extra water goes

Differencing the two clamped engines isolates the wave-setup response. The extra
water is **not** spread evenly: it piles into the back-bay marshes and the estuary
shoreline, which is exactly where the model has been under-filling against the HWMs.

In [ ]:
plots.plot_engine_difference(
    "snapwave_tuned_25m", "galibier_gammax2_25m",
    label_a="Faber", label_b="Galibier+clamp",
);

### Why a small ocean-side difference makes a large estuary difference

Galibier's wave *field* is not bigger — its `hm0` is 5–15 % **smaller** than Faber's
almost everywhere. Setup is driven by the gradient of radiation stress (i.e. by how
energy is **dissipated**), so Galibier's v2.4.0 breaking rework converts a slightly
weaker wave field into far more momentum flux.

The estuary itself is near-calm (mean `hm0` ≈ 0.28 m); +1.7 m of setup cannot be
generated there locally. It is *imported* over the Sea Bright barrier — and the
barrier sits **exactly at the threshold of being overtopped**.

What governs flow across the barrier is not the cell-average `zb` (1.95 m — misleading)
but the **subgrid u/v face table**. Taking, for each shore-normal transect, the highest
`uv_zmin` a water parcel must climb gives the *controlling sill*:

| | elevation | % of barrier overtopped |
|---|---|---|
| **median barrier sill** (subgrid `uv_zmin`) | **3.56 m** | — |
| Faber ocean-side peak | 3.64 m | **59 %** |
| Galibier ocean-side peak | 3.88 m | **75 %** |

The storm tide lands on the sill almost exactly. That knife-edge is the amplifier:
+0.23 m of extra setup lengthens the overtopped section AND deepens the head over it,
which together drive **+40 % estuary storage** (49.2 → 68.9 Mm³) and **+1.16 m** of
estuary level.

The subgrid therefore *does* preserve the revetment — the barrier is not smoothed away.
Which leaves a sharper suspect for the residual under-fill: the DEM is **pre-storm (2010
lidar) and static**, while the real barrier eroded as the surge peaked. A fixed sill held
at threshold for 48 h will systematically under-deliver, and no hydraulic knob can fix it.